<a href="https://colab.research.google.com/github/Benitmulindwa/Cheminformatics/blob/main/QSAR_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install rdkit pandas datamol molfeat numpy scikit-learn yellowbrick wget

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.5/567.5 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [10]:
import pandas as pd

In [11]:
data = {
    'smiles': [
        'CCO', 'CC(=O)Oc1ccccc1C(=O)O', 'c1ccccc1', 'CC(=O)O', 'C1CCCCC1',
        'CN1C=NC2=C1C(=O)N(C(=O)N2C)C', 'COc1cc(C=O)ccc1O', 'CC1=CC=C(C=C1)O',
        'C1=CC=C(C=C1)O', 'ClC1=CC=CC=C1', 'OC1=CC=CC=C1', 'CCN(CC)CC'
    ],
    # Dummy activity values (e.g., pIC50)
    'activity': [0.5, 4.2, 2.1, 1.1, 2.5, 3.8, 3.1, 2.9, 2.8, 3.2, 2.7, 1.5]
}

In [19]:
df=pd.DataFrame(data)
df

,smiles,activity
0,CCO,0.5
1,CC(=O)Oc1ccccc1C(=O)O,4.2
2,c1ccccc1,2.1
3,CC(=O)O,1.1
4,C1CCCCC1,2.5
5,CN1C=NC2=C1C(=O)N(C(=O)N2C)C,3.8
6,COc1cc(C=O)ccc1O,3.1
7,CC1=CC=C(C=C1)O,2.9
8,C1=CC=C(C=C1)O,2.8
9,ClC1=CC=CC=C1,3.2


In [16]:
from rdkit.Chem import AllChem, rdFingerprintGenerator
import numpy as np
from rdkit import DataStructs

In [17]:
def morgan_fingerprint(mol):
  mol=AllChem.MolFromSmiles(mol)
  if mol is None:
    return None
  generator=rdFingerprintGenerator.GetMorganGenerator(2,fpSize=2048)
  fp=generator.GetFingerprint(mol)
  arr=np.array(fp)
  DataStructs.ConvertToNumpyArray(fp,arr)
  return arr

In [23]:
# Apply the function to create the X matrix
X = np.array([morgan_fingerprint(s) for s in df['smiles']])
y = np.array(df['activity'])

In [25]:
from sklearn.model_selection import train_test_split

In [26]:
# 3. SPLIT DATA
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [27]:
from sklearn.ensemble import RandomForestRegressor

In [28]:
# 4. BUILD and TRAIN MODEL
# (Using Random Forest)
rf_model=RandomForestRegressor(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)

RandomForestRegressor(random_state=42)

In [31]:
from sklearn.metrics import mean_squared_error, r2_score

In [35]:
# 5.EVALUATION

y_pred=rf_model.predict(X_test)
np.sqrt(mean_squared_error(y_test,y_pred))

np.float64(0.8462072244235834)

In [36]:
print(f"SCORE: {r2_score(y_test,y_pred):.4f}")

SCORE: 0.4794


In [38]:
# Example Prediction
new_mol = "CC(=O)Nc1ccc(O)cc1" # Paracetamol
new_fp = morgan_fingerprint(new_mol).reshape(1, -1)
prediction = rf_model.predict(new_fp)
print(f"\nPredicted Activity for Paracetamol: {prediction[0]:.2f}")


Predicted Activity for Paracetamol: 2.99
